# MNIST MLP Inference on PYNQ-Z2
Run this notebook **on the PYNQ-Z2 board** (Jupyter at 192.168.2.99)

Requirements:
- `mlp.bit` + `mlp.hwh` in the same directory as this notebook
- `fpga_weights/` folder (w1.npy … b4.npy) in the same directory
- MNIST test set available (downloads automatically)


In [ ]:
import numpy as np
import time
from pynq import Overlay, allocate
import pynq.lib.dma
from PIL import Image

# Download MNIST test set for evaluation
import urllib.request, gzip, os
def download_mnist():
    base = 'http://yann.lecun.com/exdb/mnist/'
    files = ['t10k-images-idx3-ubyte.gz', 't10k-labels-idx1-ubyte.gz']
    for f in files:
        if not os.path.exists(f[:-3]):
            print(f'Downloading {f}...')
            urllib.request.urlretrieve(base + f, f)
            with gzip.open(f) as gz, open(f[:-3],'wb') as out:
                out.write(gz.read())
    imgs  = np.frombuffer(open('t10k-images-idx3-ubyte','rb').read(), np.uint8, offset=16).reshape(-1,784)
    labels= np.frombuffer(open('t10k-labels-idx1-ubyte','rb').read(), np.uint8, offset=8)
    return imgs, labels

imgs, labels = download_mnist()
print(f'Loaded {len(imgs)} test images')


In [ ]:
# Load the FPGA overlay (bitstream + hardware handoff)
print('Loading overlay...')
ol = Overlay('./mlp.bit')
print('Overlay loaded.')
print(ol.ip_dict.keys())  # Should list mlp_top_0 and axi_dma_0

dma = ol.axi_dma_0
mlp = ol.mlp_top_0


In [ ]:
# Load quantised weights (int16) and allocate contiguous DDR buffers
# The MLP HLS IP reads these via AXI4-HP0
SCALE = 1024   # matches quantisation scale in export script

weight_bufs = {}
bias_bufs   = {}

for k in range(1, 5):
    w = np.load(f'fpga_weights/w{k}.npy')  # int16, shape (out, in)
    b = np.load(f'fpga_weights/b{k}.npy')  # int16, shape (out,)

    # Allocate physically contiguous buffers that the PL can DMA from
    w_buf = allocate(shape=w.shape, dtype=np.int16)
    b_buf = allocate(shape=b.shape, dtype=np.int16)
    w_buf[:] = w
    b_buf[:] = b
    weight_bufs[k] = w_buf
    bias_bufs[k]   = b_buf
    print(f'Layer {k}: W{w.shape} B{b.shape}  '
          f'physical addr W=0x{w_buf.physical_address:08X}')

# Write weight buffer physical addresses to MLP AXI4-Lite control registers
# Register map is defined by the HLS interface (s_axilite) — check IP doc
# Typical offsets (check mlp_top/solution/impl/ip/drivers/mlp_top_v1_0/src/):
REG = {
    'w1': 0x10, 'b1': 0x18,
    'w2': 0x20, 'b2': 0x28,
    'w3': 0x30, 'b3': 0x38,
    'w4': 0x40, 'b4': 0x48,
}

for k in range(1, 5):
    mlp.write(REG[f'w{k}'], weight_bufs[k].physical_address & 0xFFFFFFFF)
    mlp.write(REG[f'b{k}'], bias_bufs[k].physical_address   & 0xFFFFFFFF)

print('Weight addresses written to HLS IP registers.')


In [ ]:
# Inference function: send one image, get 10 logits
def preprocess(img_uint8):
    """Normalise MNIST uint8 image to float, then quantise to int16 (scale=1024)."""
    x = img_uint8.astype(np.float32) / 255.0
    # MNIST normalisation: mean=0.1307, std=0.3081
    x = (x - 0.1307) / 0.3081
    # Quantise to fixed-point int16
    x_q = np.clip(np.round(x * SCALE), -32768, 32767).astype(np.int16)
    return x_q

# Allocate I/O buffers (contiguous, required for DMA)
in_buf  = allocate(shape=(784,), dtype=np.int16)
out_buf = allocate(shape=(10,),  dtype=np.int16)

def fpga_inference(img_uint8):
    """Run one MNIST image through the FPGA MLP accelerator."""
    in_buf[:] = preprocess(img_uint8)

    # Trigger MLP start via AXI4-Lite (write 1 to AP_START register at offset 0x00)
    mlp.write(0x00, 1)

    # DMA: send input (mm2s), receive output (s2mm)
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

    # Convert int16 logits back to float (undo scale factor)
    logits = out_buf[:].astype(np.float32) / SCALE
    return logits

print('Inference function ready.')


In [ ]:
# Test a single image
idx = 0
logits = fpga_inference(imgs[idx])
pred   = np.argmax(logits)

print(f'Image index  : {idx}')
print(f'True label   : {labels[idx]}')
print(f'Predicted    : {pred}')
print(f'Logits       : {np.round(logits, 2)}')
print(f'Result       : {"CORRECT" if pred == labels[idx] else "WRONG"}')

# Display the image
import matplotlib.pyplot as plt
plt.figure(figsize=(3,3))
plt.imshow(imgs[idx].reshape(28,28), cmap='gray')
plt.title(f'True: {labels[idx]}  Pred: {pred}')
plt.axis('off'); plt.show()


In [ ]:
# Full accuracy benchmark on 1000 test images + throughput measurement
N = 1000
correct = 0

t0 = time.time()
for i in range(N):
    logits = fpga_inference(imgs[i])
    if np.argmax(logits) == labels[i]:
        correct += 1
elapsed = time.time() - t0

acc        = correct / N * 100
throughput = N / elapsed

print(f'=== FPGA Benchmark ({N} images) ===')
print(f'Accuracy    : {acc:.2f}%')
print(f'Total time  : {elapsed:.2f}s')
print(f'Throughput  : {throughput:.1f} images/sec')
print(f'Latency/img : {elapsed/N*1000:.1f} ms')

if acc >= 96.0:
    print('SUCCESS: FPGA accuracy within 2% of software baseline (98.44%)')
else:
    print('WARN: Check quantisation precision or weight register addresses')


In [ ]:
# Free contiguous memory buffers
for k in range(1,5):
    weight_bufs[k].freebuffer()
    bias_bufs[k].freebuffer()
in_buf.freebuffer()
out_buf.freebuffer()
print('Buffers freed.')
